In [2]:
import pandas as pd

import heapq

from math import radians, sin, cos, sqrt, asin

In [3]:
df = pd.read_csv("cleaned_datase222t.csv")
df

,Place_Name,Latitude,Longitude,distance,traffic,time,fast_route,fuel_consumption,vehicle_load_kg,road_type,weather,road_condition,hour_of_day
0,"Graphic Era Deemed to be University, Dehradun",30.2685,77.9968,23.72,1,39.39,1,0.111,1000,Highway,Rainy,Good,14
1,ISBT Dehradun,30.2861,77.9954,57.14,1,91.19,1,0.123,3000,City Road,Clear,Good,16
2,"Clement Town, Dehradun",30.2662,78.0075,44.46,1,73.68,1,0.119,2000,City Road,Rainy,Good,8
3,"Sahranpur Road, Dehradun",30.2988,77.9945,36.72,4,87.46,1,0.218,1000,Rural Road,Clear,Bad,18
4,"Clock Tower, Dehradun",30.3244,78.0411,11.05,5,64.08,0,0.243,500,Highway,Rainy,Bad,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2146,Unknown Location,26.9911,75.7611,2.68,2,25.86,0,0.147,1000,Highway,Clear,Good,20
2147,Madina Crossing,28.9711,76.4511,26.48,1,43.59,1,0.111,2000,Rural Road,Rainy,Good,19
2148,Bageshwar,29.8415,79.7712,19.11,1,26.73,1,0.114,3000,Highway,Clear,Good,8
2149,Kashipur,29.2104,78.9618,35.48,2,65.18,1,0.155,1000,City Road,Clear,Good,15


In [4]:
def haversine(lat1, lon1, lat2, lon2):

    R = 6371

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1

    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2

    return R * 2 * asin(sqrt(a))

In [5]:
# ================= BUILD GRAPH =================

def build_graph(df, max_neighbors=8):

    graph = {row["Place_Name"]: [] for _, row in df.iterrows()}

    coords = {
        row["Place_Name"]: (row["Latitude"], row["Longitude"])
        for _, row in df.iterrows()
    }

    # Fast lookup
    place_lookup = df.groupby("Place_Name").first().to_dict("index")

    for _, row1 in df.iterrows():
        place1 = row1["Place_Name"]
        lat1, lon1 = coords[place1]

        distances = []

        for _, row2 in df.iterrows():
            place2 = row2["Place_Name"]

            if place1 != place2:
                lat2, lon2 = coords[place2]
                d = haversine(lat1, lon1, lat2, lon2)
                distances.append((d, place2))

        distances.sort()

        for d, place2 in distances[:max_neighbors]:

            row2 = place_lookup[place2]

            edge_data = {
                "distance": d,
                "traffic": row2["traffic"],
                "time": row2["time"],
                "fuel": row2["fuel_consumption"],
                "load": row2["vehicle_load_kg"],
                "road_type": row2["road_type"],
                "weather": row2["weather"],
                "road_condition": row2["road_condition"],
                "hour": row2["hour_of_day"]
            }

            graph[place1].append((place2, edge_data))

    return graph

In [6]:
def compute_cost(edge):

    w = {
        "distance": 1,
        "traffic": 2,
        "time": 1.5,
        "fuel": 1,
        "load": 0.5
    }

    cost = (
        w["distance"] * edge["distance"] +
        w["traffic"] * edge["traffic"] +
        w["time"] * edge["time"] +
        w["fuel"] * edge["fuel"] +
        w["load"] * edge["load"]
    )

    return cost

In [7]:
# ================= HEURISTIC =================

def heuristic(df, node, goal):

    row1 = df[df["Place_Name"] == node].iloc[0]

    row2 = df[df["Place_Name"] == goal].iloc[0]

    return haversine(

        row1["Latitude"], row1["Longitude"],

        row2["Latitude"], row2["Longitude"]

    )

In [8]:
# ================= A* =================

def astar(graph, df, start, goal):

    open_set = []
    heapq.heappush(open_set, (0, start))

    g_score = {node: float("inf") for node in graph}
    g_score[start] = 0

    came_from = {}


    meta = {
        node: {"dist": 0, "traffic": 0, "time": 0, "fuel": 0}
        for node in graph
    }

    while open_set:

        _, current = heapq.heappop(open_set)

        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            path.reverse()

            return path, meta[goal]

        for neighbor, edge in graph[current]:

            cost = compute_cost(edge)
            tentative = g_score[current] + cost

            if tentative < g_score[neighbor]:

                came_from[neighbor] = current
                g_score[neighbor] = tentative

                
                meta[neighbor]["dist"] = meta[current]["dist"] + edge["distance"]
                meta[neighbor]["traffic"] = meta[current]["traffic"] + edge["traffic"]
                meta[neighbor]["time"] = meta[current]["time"] + edge["time"]
                meta[neighbor]["fuel"] = meta[current]["fuel"] + edge["fuel"]

                f = tentative + heuristic(df, neighbor, goal)
                heapq.heappush(open_set, (f, neighbor))

    return None, None

In [9]:
def top3_paths(graph, df, start, goal):

    paths = []
    excluded_edges = set()

    for _ in range(3):

        # Create temp graph
        temp_graph = {}

        for node, edges in graph.items():
            temp_graph[node] = [
                (nbr, edge) for (nbr, edge) in edges
                if (node, nbr) not in excluded_edges
            ]

        path, meta = astar(temp_graph, df, start, goal)

        if path is None:
            break

        paths.append((path, meta))

        
        for i in range(len(path) - 1):
            excluded_edges.add((path[i], path[i+1]))

    return paths

In [64]:
def main():

    graph = build_graph(df)

    print("\n===== ROUTE OPTIMIZATION SYSTEM =====")


    start = input("\nEnter start place: ").strip()
    goal = input("Enter goal place: ").strip()

    if start not in graph or goal not in graph:
        print(" Invalid location")
        return

    print("\nChoose Mode:")
    print("1. Single Optimal Route (A*)")
    print("2. Alternative Optimal Routes (Top-3 A*)")

    choice = input("Enter choice: ")

    # SINGLE PATH
    if choice == "1":

        path, meta = astar(graph, df, start, goal)

        if path is None:
            print(" No path found")
            return

        print("\n===== BEST ROUTE =====")
        print(" → ".join(path))
        print("Distance:", round(meta["dist"], 2))
        print("Traffic:", round(meta["traffic"], 2))
        print("Time:", round(meta["time"], 2))
        print("Fuel:", round(meta["fuel"], 2))

    # TOP 3 PATHS
    elif choice == "2":

        paths = top3_paths(graph, df, start, goal)

        if not paths:
            print(" No path found")
            return

        print("\n===== ALTERNATIVE OPTIMAL ROUTES =====")

        for i, (path, meta) in enumerate(paths, 1):

            print(f"\n--- Route {i} ---")
            print("Path:", " → ".join(path))
            print("Distance:", round(meta["dist"], 2))
            print("Traffic:", round(meta["traffic"], 2))
            print("Time:", round(meta["time"], 2))
            print("Fuel:", round(meta["fuel"], 2))

    else:
        print(" Invalid choice")

In [1]:
main()

NameError: name 'main' is not defined

In [12]:
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("(", "").str.replace(")", "")

In [13]:
X = df[[
    "distance",
    "vehicle_load_kg",
    "road_type",
    "weather",
    "road_condition",
    "hour_of_day"
]]

In [15]:
X = pd.get_dummies(X)#To handle catogorial data

In [16]:
y_traffic = df["traffic"]
y_time = df["time"]
y_fuel = df["fuel_consumption"]

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, yT_train, yT_test = train_test_split(X, y_traffic, test_size=0.2, random_state=42)
_, _, ytime_train, ytime_test = train_test_split(X, y_time, test_size=0.2, random_state=42)
_, _, yfuel_train, yfuel_test = train_test_split(X, y_fuel, test_size=0.2, random_state=42)

In [18]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, yT_train)
pred_lr = lr.predict(X_test)

In [19]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=3)
knn.fit(X_train, yT_train)
pred_knn = knn.predict(X_test)

In [20]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor()
dt.fit(X_train, yT_train)
pred_dt = dt.predict(X_test)

In [21]:
from sklearn.linear_model import LogisticRegression

y_class = (y_traffic > y_traffic.mean()).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_class, test_size=0.2)

log = LogisticRegression(max_iter=1000)
log.fit(X_train_c, y_train_c)
pred_log = log.predict(X_test_c)

In [22]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3)
df["cluster"] = kmeans.fit_predict(X)

In [2]:
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

print("\n===== MODEL COMPARISON (TRAFFIC) =====")

print("Linear Regression MAE:", mean_absolute_error(yT_test, pred_lr))
print("KNN MAE:", mean_absolute_error(yT_test, pred_knn))
print("Decision Tree MAE:", mean_absolute_error(yT_test, pred_dt))

print("Logistic Accuracy:", accuracy_score(y_test_c, pred_log))


===== MODEL COMPARISON (TRAFFIC) =====


NameError: name 'yT_test' is not defined

In [57]:
]

NameError: name 'graph' is not defined